In [ ]:
# ===== Ô COLAB (thêm bởi _bundles/colab_tu_kaggle.py) — notebook gốc: uyenle0712/sa3-grpo-c-2 =====
# Cần: Runtime GPU (A100 nếu chạy hai tiến trình), Colab secret KAGGLE_UYENLE0712 = khoá KGAT_ của tài khoản Kaggle "uyenle0712".
import os, subprocess, sys, glob, time, threading, shutil, json
import subprocess as _sp
class _Popen(_sp.Popen):
    def __init__(self, *a, **k):
        k.setdefault("start_new_session", True); super().__init__(*a, **k)
_sp.Popen = _Popen
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):   # chạy qua Colab CLI thì gắn trước bằng `colab drivemount`
    drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/thesis/colab_kaggle/sa3-grpo-c-2"
os.makedirs(DRIVE, exist_ok=True); os.makedirs("/kaggle/input", exist_ok=True); os.makedirs("/kaggle/working", exist_ok=True)
SEC = "KAGGLE_UYENLE0712"   # Colab secret: khoá KGAT_ của tài khoản uyenle0712
TOKF = "/content/.kaggle_tok/access_token"   # Colab CLI: `colab upload` tệp khoá lên đây (KAGGLE_API_TOKEN nhận đường dẫn tệp)
if os.path.isfile(TOKF): os.environ["KAGGLE_API_TOKEN"] = TOKF
else:
  try: os.environ["KAGGLE_API_TOKEN"] = userdata.get(SEC)
  except Exception: raise SystemExit(f"DUNG: them Colab secret {SEC} (khoa KGAT_ cua tai khoan uyenle0712) va bat 'Notebook access'")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle>=2.2.4"], check=True)
def tai(args):
    t0 = time.time()
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    print(" ".join(args[:3]), "· ma", r.returncode, f"· {time.time()-t0:.0f} s", (r.stdout + r.stderr)[-300:].replace(os.environ["KAGGLE_API_TOKEN"], "***"), flush=True)
    assert r.returncode == 0, "DUNG: tai tu Kaggle loi (sai tai khoan? can tai khoan uyenle0712)"
tai(["datasets", "download", "-d", "uyenle0712/sa3-grpo-v3", "-p", "/kaggle/input/sa3-grpo-v3", "--unzip"])
tai(["datasets", "download", "-d", "uyenle0712/grpo-luoi-script", "-p", "/kaggle/input/grpo-luoi-script", "--unzip"])
tai(["kernels", "output", "uyenle0712/sa3-grpo-u-1", "-p", "/kaggle/input/sa3-grpo-u-1"])
print(subprocess.run("du -sh /kaggle/input/* ; nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; df -h / | tail -1", shell=True, capture_output=True, text=True).stdout, flush=True)


In [ ]:
import subprocess, sys, os, time, glob, json, shutil, hashlib, re, ast, statistics as st
T_NB = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java")
sh("java -version")
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' 'trl==0.29.1' accelerate pycocoevalcap sacrebleu -U torchao 2>&1 | tail -3")
import trl, transformers, peft, torch
print(trl.__version__, transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), flush=True)
assert (trl.__version__, transformers.__version__, peft.__version__) == ("0.29.1", "5.20.2", "0.21.2"), "DUNG: lech phien ban"
assert torch.cuda.device_count() >= 1, "DUNG: can 2 GPU T4"


In [ ]:
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
D = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/sa3_grpo.py", recursive=True) if "/notebooks/" not in p and os.path.isdir(os.path.join(os.path.dirname(p), "sft_final"))]
assert len(D) == 1, D; D = D[0]
for f, h in {'sa3_grpo.py': 'de70891e11c2b03a9cf035bd79b6122c', 'grpo_luoi.py': '92d0a9bc6b8cae8a480aef08f99fb9d8', 'grpo_spice.py': '8489108147c8d1ed80c3dc67c8ad3d3f', 'build_branch_data.py': '619e63e123a6dbf60086e65ee94a3912', 'sa3_reward_screen.py': 'f6637048ed5dce6db051b4fabf15833d', 'sa3_action_verify.py': '672e2b6ff04155a7214e8880509089a2', 'h4_spice_tuple_7_11.py': 'd5f13d93fb84235f60c4bd2e558a54a5', 'train_v3.jsonl': 'e033ea809a18c0e5e3eb878305396f19', 'mau_0.jsonl': '738090ee6133a24a0923e3b8cca326d3', 'ocr.jsonl': 'f934eede0a73b571d85102884dc8e7f0'}.items():
    assert md5(f"{D}/{f}") == h, f"DUNG: {f} lech ban may nha"
for f in ('sa3_grpo.py', 'grpo_luoi.py', 'grpo_spice.py', 'build_branch_data.py', 'sa3_reward_screen.py', 'sa3_action_verify.py', 'h4_spice_tuple_7_11.py'):
    shutil.copy(f"{D}/{f}", f"{W}/{f}")
import importlib.util
SPL = list(importlib.util.find_spec("pycocoevalcap").submodule_search_locations)[0] + "/spice/lib"
for j, sz in {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}.items():
    dst = f"{SPL}/{j}"
    if os.path.exists(dst) and os.path.getsize(dst) == sz: continue
    src = [p for p in glob.glob(f"/kaggle/input/**/{j}", recursive=True) if os.path.getsize(p) == sz]; assert src, f"DUNG: thieu {j}"
    shutil.copy(src[0], dst)
def py(*a):
    r = subprocess.run(["python", *a], cwd=W, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-2500:], flush=True); return r.returncode
assert py("sa3_grpo.py", "--selftest") == 0, "DUNG: selftest sa3_grpo"
assert py("grpo_luoi.py", "--selftest-algo") == 0, "DUNG: selftest algo"
ARMS, SEED, TAG = ['SA-SPICE-v3', 'chrF'], 101, {'SA-SPICE-v3': 'saspicev3', 'chrF': 'chrf'}
OUT = {a: f"{W}/sa3_{TAG[a]}_{SEED}" for a in ARMS}
for a, o in OUT.items():
    os.makedirs(o, exist_ok=True)
    ck = [p for p in glob.glob(f"/kaggle/input/**/{os.path.basename(o)}/checkpoint-*", recursive=True) if os.path.isdir(p)]
    for p in ck:
        shutil.copytree(p, os.path.join(o, os.path.basename(p)), dirs_exist_ok=True)
    print(a, "→", o, "· diem luu da chep:", sorted(os.path.basename(p) for p in ck), flush=True)
    assert ck or not True, f"DUNG: co kernel truoc ma khong thay diem luu cua {o}"
S1M, SFTM = "/tmp/s1_merged", "/tmp/sft_cont_merged"
print(sh("df -h /tmp /kaggle/working | tail -2"))
assert py("grpo_spice.py", "--merge", "--bundle", D, "--adapter", f"{D}/adapter_s1_seed101", "--merged", S1M) == 0, "DUNG: hoa S1"
assert py("sa3_grpo.py", "--hoa", "--data", D, "--merged-s1", S1M, "--merged", SFTM) == 0, "DUNG: hoa SFT-cont"
shutil.rmtree(S1M, ignore_errors=True)
KH = py("sa3_grpo.py", "--kiem-hoa", "--data", D, "--merged", SFTM)
if KH != 0:
    r_ = subprocess.run(["python", "sa3_grpo.py", "--kiem-hoa", "--data", D, "--merged", SFTM], cwd=W, capture_output=True, text=True).stdout
    k_ = re.findall(r"trùng kernel 1: (\d+)/(\d+)", r_)
    assert k_ and int(k_[-1][0]) >= int(k_[-1][1]) - 1, "DUNG: kiem hoa lech qua 1 cau"
    print("⚠️ COLAB: kiem hoa lech", k_[-1], "— GPU khac T4, chap nhan lech 1 cau, GHI VAO BAO CAO", flush=True)
print("✅ moi truong + hoa + kiem hoa", flush=True)


In [ ]:
# ===== Ô COLAB: chạy tiếp sau mất máy + đồng bộ Drive mỗi 5 phút =====
if os.listdir(DRIVE):
    print("khoi phuc tu Drive:", subprocess.run(f"rsync -a '{DRIVE}/' /kaggle/working/ && du -sh /kaggle/working", shell=True, capture_output=True, text=True).stdout, flush=True)
def _dong_bo():
    while True:
        time.sleep(300)
        subprocess.run(f"rsync -a --exclude '*.zip' /kaggle/working/ '{DRIVE}/'", shell=True)
threading.Thread(target=_dong_bo, daemon=True).start()
print("✅ dong bo Drive moi 5 phut →", DRIVE, flush=True)


In [ ]:
CAT = 3600 * 23.0
P = []
for g, a in enumerate(ARMS):
    o = OUT[a]
    cmd = ["python", "sa3_grpo.py", "--train", "--arm", a, "--seed", str(SEED), "--data", D, "--merged", SFTM, "--out", o,
           "--no-q4", "--gen-chunk", "8", "--bs", "4", "--accum", "4", "--max-steps", "500", "--resume", "auto"]
    P.append((a, o, subprocess.Popen(cmd, cwd=W, stdout=open(f"{o}/train.log", "a"), stderr=subprocess.STDOUT, text=True,
        env={**os.environ, "CUDA_VISIBLE_DEVICES": "0", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
             "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})))
    time.sleep(90)
def cks(o):
    return sorted(glob.glob(f"{o}/checkpoint-*"), key=lambda s: int(s.rsplit("-", 1)[1]))
def tinh_trang(a, o):
    L = open(f"{o}/train.log", encoding="utf-8", errors="ignore").read().splitlines()
    tiep = [l for l in L if l.startswith("[tiếp từ]")]
    goc = int(re.findall(r"checkpoint-(\d+)", tiep[-1])[0]) if tiep and "checkpoint-" in tiep[-1] else 0
    Dl = []
    for l in L:
        if l.startswith("{'loss'"):
            try: Dl.append({k: float(v) for k, v in ast.literal_eval(l).items() if re.match(r"^[-\d.e+na]+$", str(v))})
            except Exception: pass
    c = cks(o)
    msg = f"  [{a}] buoc {goc + len([d for d in Dl if 'reward' in d])}/500 (tiep tu {goc}) · luu {os.path.basename(c[-1]) if c else 'chua co'}"
    x = [d for d in Dl if "reward" in d][-10:]
    if x:
        msg += f" · {st.mean(d.get('step_time', 0) for d in Dl[-20:]):.0f} s/lan · thuong {st.mean(d['reward'] for d in x):.3f} · kl {st.mean(d.get('kl', 0) for d in x):.4f}"
    for k in ("[xong]", "[nạp default]", "[LoRA mới]", "[câu nhắc]"):
        y = [l for l in L if l.startswith(k)]
        if y: msg += "\n      " + y[-1][:150]
    if any("Traceback" in l or "OutOfMemory" in l or "Killed" in l or "STOP" in l for l in L[-80:]): msg += "\n      ⚠️ Traceback/OOM/STOP o cuoi log"
    return msg
while any(p.poll() is None for _, _, p in P):
    time.sleep(120)
    for _, o, _ in P:
        c = cks(o)
        for x in c[:-2]:
            if int(x.rsplit("-", 1)[1]) % 100:
                shutil.rmtree(x, ignore_errors=True); print("  [xoa diem luu]", os.path.basename(x), flush=True)
    m = subprocess.run("free -g | awk '/Mem/{print $3\"/\"$2\" GB RAM\"}'; nvidia-smi --query-gpu=memory.used --format=csv,noheader | tr '\\n' ' '",
                       shell=True, capture_output=True, text=True).stdout.strip()
    print(f"{(time.time()-T_NB)/3600:5.2f} h · {m}", flush=True)
    for a, o, p in P: print(tinh_trang(a, o), flush=True)
    if time.time() - T_NB > CAT:
        print("⛔ het gio, dung de Output kip luu", flush=True)
        for _, _, p in P:
            if p.poll() is None: p.terminate()
        for _, _, p in P: p.wait()
for a, o, p in P:
    c = cks(o)
    print(f"\n===== {a} · ma thoat {p.returncode} · diem luu {[os.path.basename(x) for x in c]}")
    print("\n".join(l for l in open(f"{o}/train.log", encoding="utf-8", errors="ignore").read().splitlines()[-25:]
                    if not l.startswith(("PTB", "Parsing", "Threads", "Loading", "[main]"))))
shutil.rmtree(SFTM, ignore_errors=True)


In [ ]:
# ===== Ô COLAB: đồng bộ lần chót =====
print(subprocess.run(f"rsync -a /kaggle/working/ '{DRIVE}/' && ls -la '{DRIVE}' | tail -20", shell=True, capture_output=True, text=True).stdout, flush=True)
